# NOTEARS experiment analysis

In [1]:
from pathlib import Path
import json
import warnings
import re

import pandas as pd

## grid search

In [6]:
INPUT_DIRS = {
    "linear": Path("grid_search"),
}

ACC_KEYS = {
    "acc_prior_ll": ("with_prior", "likelihood"),
    "acc_prior_l2": ("with_prior", "l2"),
    "acc_no_prior_ll": ("no_prior", "likelihood"),
    "acc_no_prior_l2": ("no_prior", "l2"),
}

In [ ]:
def load_accuracy_rows(input_dirs):
    rows = []

    for experiment, directory in input_dirs.items():
        for path in sorted(directory.glob("*.json")):
            with path.open(encoding="utf-8") as file:
                result = json.load(file)

            d = len(result.get("B_true", [])) or None
            seed_match = re.search(r"_seed(\d+)(?=_|\.json$)", path.name)
            seed = int(seed_match.group(1)) if seed_match else None

            epsilon_match = re.search(r"_epsilon([^_]+)(?=_seed)", path.stem)
            epsilon = (
                float(epsilon_match.group(1))
                if epsilon_match else float("nan")
            )

            configuration = re.sub(r"_seed\d+(?=_|$)", "", path.stem)

            for key, (prior_setting, loss) in ACC_KEYS.items():
                scores = result.get(key)
                if not isinstance(scores, dict):
                    continue

                row = {
                    "experiment": experiment,
                    "prior_setting": prior_setting,
                    "loss": loss,
                    "d": d,
                    "seed": seed,
                    "epsilon": epsilon,
                    "configuration": configuration,
                }
                row.update({
                    name: value
                    for name, value in scores.items()
                    if isinstance(value, (int, float))
                })
                rows.append(row)

    return pd.DataFrame(rows)


accuracy_rows = load_accuracy_rows(INPUT_DIRS)
accuracy_rows.head()

In [ ]:
metadata_columns = {
    "experiment", "configuration", "prior_setting",
    "loss", "d", "seed", "epsilon", "file",
}
score_columns = [
    column for column in accuracy_rows.columns
    if column not in metadata_columns
    and pd.api.types.is_numeric_dtype(accuracy_rows[column])
]

# Average runs within each setup; f1_std is sample standard deviation (ddof=1).
summary = (
    accuracy_rows
    .groupby(
        ["experiment", "d", "epsilon", "prior_setting", "loss"],
        dropna=False,
    )
    .agg(num_seeds=("seed", "nunique"), f1_std=("f1", "std"), **{
        score: (score, "mean") for score in score_columns
    })
    .round(4)
)
summary

### Conclusion: epsilon 0.1 is the best

## Exist Edges

In [17]:
# Change these paths if the result directories are elsewhere.
INPUT_DIRS = {
    "linear": Path("linear_exist_trek_pairs"),
    "nonlinear": Path("nonlinear_exist_edge_pairs"),
}

# Set this to True only if you want old files whose loss type was not saved.
INCLUDE_LEGACY_UNSPECIFIED = False

ACC_KEYS = {
    "acc_prior_ll": ("with_prior", "likelihood"),
    "acc_prior_l2": ("with_prior", "l2"),
    "acc_no_prior_ll": ("no_prior", "likelihood"),
    "acc_no_prior_l2": ("no_prior", "l2"),
}

if INCLUDE_LEGACY_UNSPECIFIED:
    ACC_KEYS.update({
        "acc_prior": ("with_prior", "unspecified"),
        "acc_no_prior": ("no_prior", "unspecified"),
    })

In [18]:
def load_accuracy_rows(input_dirs):
    rows = []

    for experiment, directory in input_dirs.items():

        for path in sorted(directory.glob("*.json")):
            with path.open(encoding="utf-8") as file:
                result = json.load(file)

            d = len(result.get("B_true", [])) or None
            seed_match = re.search(r"_seed(\d+)(?=_|\.json$)", path.name)
            seed = int(seed_match.group(1)) if seed_match else None
            epsilon_match = re.search(r"_epsilon([^_]+)(?=_seed)", path.stem)
            epsilon = float(epsilon_match.group(1)) if epsilon_match else float("nan")
            configuration = re.sub(r"_seed\d+(?=_|$)", "", path.stem)
            for key, (prior_setting, loss) in ACC_KEYS.items():
                scores = result.get(key)

                if not isinstance(scores, dict):
                    continue
                
                row = {
                    "experiment": experiment,
                    "prior_setting": prior_setting,
                    "loss": loss,
                    "d": d,
                    "seed": seed,
                    "epsilon": epsilon,
                    "configuration": configuration,
                    "file": path.name,
                }
                row.update({
                    score_name: score_value
                    for score_name, score_value in scores.items()
                    if isinstance(score_value, (int, float))
                })
                rows.append(row)

    return pd.DataFrame(rows)

accuracy_rows = load_accuracy_rows(INPUT_DIRS)

print(f"Loaded {len(accuracy_rows)} accuracy records from {accuracy_rows['file'].nunique()} JSON files.")
accuracy_rows.head().style.set_caption("Raw per-run accuracy records")

Loaded 80 accuracy records from 20 JSON files.


,experiment,prior_setting,loss,d,seed,epsilon,configuration,file,fdr,tpr,fpr,f1,shd,nnz
0,linear,with_prior,likelihood,10,0,0.010000,linear_exist_trek_pairs_ER1_d10_gauss_rate0.5_epsilon0.01_twopenalty,linear_exist_trek_pairs_ER1_d10_gauss_rate0.5_epsilon0.01_seed0_twopenalty.json,0.666667,0.400000,0.228571,0.363636,12,12
1,linear,with_prior,l2,10,0,0.010000,linear_exist_trek_pairs_ER1_d10_gauss_rate0.5_epsilon0.01_twopenalty,linear_exist_trek_pairs_ER1_d10_gauss_rate0.5_epsilon0.01_seed0_twopenalty.json,0.555556,0.400000,0.142857,0.421053,7,9
2,linear,no_prior,likelihood,10,0,0.010000,linear_exist_trek_pairs_ER1_d10_gauss_rate0.5_epsilon0.01_twopenalty,linear_exist_trek_pairs_ER1_d10_gauss_rate0.5_epsilon0.01_seed0_twopenalty.json,0.625000,0.300000,0.142857,0.333333,10,8
3,linear,no_prior,l2,10,0,0.010000,linear_exist_trek_pairs_ER1_d10_gauss_rate0.5_epsilon0.01_twopenalty,linear_exist_trek_pairs_ER1_d10_gauss_rate0.5_epsilon0.01_seed0_twopenalty.json,0.571429,0.300000,0.114286,0.352941,9,7
4,linear,with_prior,likelihood,10,1,0.010000,linear_exist_trek_pairs_ER1_d10_gauss_rate0.5_epsilon0.01_twopenalty,linear_exist_trek_pairs_ER1_d10_gauss_rate0.5_epsilon0.01_seed1_twopenalty.json,0.636364,0.400000,0.200000,0.380952,9,11


In [19]:
metadata_columns = {
    "experiment", "configuration", "prior_setting",
    "loss", "d", "seed", "epsilon", "file",
}
score_columns = [
    column for column in accuracy_rows.columns
    if column not in metadata_columns
    and pd.api.types.is_numeric_dtype(accuracy_rows[column])
]

# Average runs within each setup; f1_std is sample standard deviation (ddof=1).
summary = (
    accuracy_rows
    .groupby(
        ["experiment", "d", "epsilon", "prior_setting", "loss"],
        dropna=False,
    )
    .agg(num_seeds=("seed", "nunique"), f1_std=("f1", "std"), **{
        score: (score, "mean") for score in score_columns
    })
    .round(4)
)
summary

num_seeds  f1_std     fdr  \
experiment d  epsilon prior_setting loss                                    
linear     10 0.01    no_prior      l2                 10  0.1366  0.6132   
                                    likelihood         10  0.1940  0.5308   
                      with_prior    l2                 10  0.1127  0.6511   
                                    likelihood         10  0.1293  0.5957   
           20 0.01    no_prior      l2                 10  0.1112  0.6384   
                                    likelihood         10  0.1995  0.3475   
                      with_prior    l2                 10  0.0789  0.6411   
                                    likelihood         10  0.1912  0.3742   

                                                  tpr     fpr      f1   shd  \
experiment d  epsilon prior_setting loss                                      
linear     10 0.01    no_prior      l2          0.320  0.1543  0.3477   8.3   
                                    likelihood  0.380  0.1286  0.4182   7.8   
                      with_prior    l2          0.340  0.1857  0.3432   8.4   
                                    likelihood  0.390  0.1686  0.3933   8.9   
           20 0.01    no_prior      l2          0.315  0.0671  0.3345  18.3   
                                    likelihood  0.525  0.0318  0.5790  11.3   
                      with_prior    l2          0.355  0.0759  0.3551  19.0   
                                    likelihood  0.575  0.0400  0.5987  11.5   

                                                 nnz  
experiment d  epsilon prior_setting loss              
linear     10 0.01    no_prior      l2           8.6  
                                    likelihood   8.3  
                      with_prior    l2           9.9  
                                    likelihood   9.8  
           20 0.01    no_prior      l2          17.7  
                                    likelihood  15.9  
                      with_prior    l2          20.0  
                                    likelihood  18.3